In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import warnings

# Mute warnings for a clean terminal
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("\n" + "★"*75)
print("🏆 THE ULTIMATE KAGGLE PIPELINE: HOLY TRINITY + TIME-SLICES + XAI")
print("★"*75)

# =========================================================
# 1. CRITICAL CONSTANT & DATA LOADING
# =========================================================
HORIZON = 24  # Strict 24-Hour Day-Ahead Forecast to prevent Data Leakage

print("\n📦 1. Loading Raw Data...")
# Ensure your CSV is uploaded to Colab!
df = pd.read_csv("cleaned_energy_data_model2.csv")
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.sort_values('start_time').reset_index(drop=True)

# =========================================================
# 2. ENGINEERING HORIZON-SAFE FEATURES (NO LEAKAGE)
# =========================================================
print(f"\n🔧 2. Engineering Features (Strict HORIZON={HORIZON}h)...")
cons = df['consumption']

# A. Calendar Features (Always safe)
df['hour']        = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.dayofweek
df['month']       = df['start_time'].dt.month
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# B. Safe Lags (Must be >= HORIZON)
df['lag_24h']  = cons.shift(HORIZON)
df['lag_48h']  = cons.shift(48)
df['lag_72h']  = cons.shift(72)
df['lag_168h'] = cons.shift(168) # Same hour last week

# C. Safe Rolling Features (Anchored exactly 24h back)
shifted = cons.shift(HORIZON)
df['rolling_mean_24h']  = shifted.rolling(24).mean()
df['rolling_std_24h']   = shifted.rolling(24).std()
df['rolling_mean_168h'] = shifted.rolling(168).mean()
df['ewm_24h']           = shifted.ewm(span=24, adjust=False).mean()

# Define final features list
features_list = ['hour', 'day_of_week', 'month', 'is_weekend',
                 'lag_24h', 'lag_48h', 'lag_72h', 'lag_168h',
                 'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_168h', 'ewm_24h']

# Add prev_daily_mean/max if they exist in your CSV
if 'prev_daily_mean' in df.columns: features_list.append('prev_daily_mean')
if 'prev_daily_max' in df.columns: features_list.append('prev_daily_max')

# =========================================================
# 3. SECURE TRAIN/VAL/TEST SPLIT
# =========================================================
print("\n✂️ 3. Train/Val/Test Split & Security Audit...")
df_clean = df.dropna(subset=features_list + ['consumption']).copy()

train_mask = df_clean['start_time'].dt.year <= 2019
val_mask   = df_clean['start_time'].dt.year == 2020
test_mask  = df_clean['start_time'].dt.year == 2021

X_train_base, y_train_base = df_clean.loc[train_mask, features_list], df_clean.loc[train_mask, 'consumption']
X_val, y_val               = df_clean.loc[val_mask, features_list], df_clean.loc[val_mask, 'consumption']
X_train_full, y_train_full = df_clean.loc[train_mask | val_mask, features_list], df_clean.loc[train_mask | val_mask, 'consumption']
X_test, y_test             = df_clean.loc[test_mask, features_list], df_clean.loc[test_mask, 'consumption']
test_dates                 = df_clean.loc[test_mask, 'start_time']

assert 'consumption' not in X_train_base.columns, "🚨 FATAL LEAK: Target in features!"

# =========================================================
# 4. OPTUNA TUNING (THE HOLY TRINITY)
# =========================================================
N_TRIALS = 15 # Kept moderate for speed. Set to 30-50 for final overnight run.

print("\n🧠 4a. Tuning LightGBM...")
def obj_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 64),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    return mean_absolute_error(y_val, lgb.LGBMRegressor(**params).fit(X_train_base, y_train_base).predict(X_val))

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(obj_lgbm, n_trials=N_TRIALS)

print("🧠 4b. Tuning XGBoost...")
def obj_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist' # Fast CPU/GPU math
    }
    return mean_absolute_error(y_val, xgb.XGBRegressor(**params).fit(X_train_base, y_train_base, verbose=False).predict(X_val))

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(obj_xgb, n_trials=N_TRIALS)

print("🧠 4c. Tuning CatBoost...")
def obj_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'random_seed': 42, 'verbose': False,
        # 'task_type': 'GPU' # 🚨 UNCOMMENT THIS IF USING COLAB T4 GPU!
    }
    return mean_absolute_error(y_val, CatBoostRegressor(**params).fit(X_train_base, y_train_base).predict(X_val))

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(obj_cat, n_trials=N_TRIALS)

# =========================================================
# 5. OPTUNA TUNING: THE MASTER BLENDER
# =========================================================
print("\n⚖️ 5. Tuning Ensemble Weights (Master Blender)...")
val_lgbm = lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, n_jobs=-1, verbose=-1).fit(X_train_base, y_train_base).predict(X_val)
val_xgb  = xgb.XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1).fit(X_train_base, y_train_base, verbose=False).predict(X_val)
val_cat  = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=False).fit(X_train_base, y_train_base).predict(X_val)

def obj_weights(trial):
    w_xgb, w_lgbm, w_cat = trial.suggest_float('w_xgb', 0, 1), trial.suggest_float('w_lgbm', 0, 1), trial.suggest_float('w_cat', 0, 1)
    tot = w_xgb + w_lgbm + w_cat
    blended = ((w_xgb/tot) * val_xgb) + ((w_lgbm/tot) * val_lgbm) + ((w_cat/tot) * val_cat)
    return mean_absolute_error(y_val, blended)

study_weights = optuna.create_study(direction='minimize')
study_weights.optimize(obj_weights, n_trials=30)
bw = study_weights.best_params
tot = bw['w_xgb'] + bw['w_lgbm'] + bw['w_cat']
W_XGB, W_LGBM, W_CAT = bw['w_xgb']/tot, bw['w_lgbm']/tot, bw['w_cat']/tot
print(f"✅ Final Ratios -> XGB: {W_XGB*100:.1f}% | LGBM: {W_LGBM*100:.1f}% | CAT: {W_CAT*100:.1f}%")

# =========================================================
# 6. FINAL TRAINING ON FULL DATA
# =========================================================
print("\n🔥 6. Final Training & Predicting 2021...")
final_lgbm = lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, n_jobs=-1, verbose=-1).fit(X_train_full, y_train_full)
final_xgb  = xgb.XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1).fit(X_train_full, y_train_full, verbose=False)
final_cat  = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=False).fit(X_train_full, y_train_full)

test_preds = (W_XGB * final_xgb.predict(X_test)) + (W_LGBM * final_lgbm.predict(X_test)) + (W_CAT * final_cat.predict(X_test))
actuals = y_test.values

# Save Submission CSV
submission = pd.DataFrame({'timestamp': test_dates, 'actual': actuals, 'ensemble_pred': test_preds})
submission.to_csv("FINAL_SUBMISSION_2021.csv", index=False)

# =========================================================
# 7. TIME-SLICED EVALUATION METRICS
# =========================================================
def print_metrics(name, y_true, y_pred):
    mae, mape, r2 = mean_absolute_error(y_true, y_pred), mean_absolute_percentage_error(y_true, y_pred), r2_score(y_true, y_pred)
    print(f"{name:<18} | MAE: {mae:<7.2f} | MAPE: {mape*100:<5.2f}% | R²: {r2:.4f}")

print("\n" + "="*75)
print("🏆 TIME-SLICED LEADERBOARD (2021 TEST SET)")
print("="*75)
print_metrics("First 24 Hours", actuals[:24], test_preds[:24])
print_metrics("First 1 Week",   actuals[:168], test_preds[:168])
print_metrics("First 1 Month",  actuals[:720], test_preds[:720])
print("-" * 75)
print_metrics("FULL YEAR 2021", actuals, test_preds)
print("="*75)

# =========================================================
# 8. VISUALIZATIONS & EXPLAINABLE AI (SHAP)
# =========================================================
print("\n📸 8. Generating Final Presentation Assets...")

# A. Feature Importance Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lgb.plot_importance(final_lgbm, ax=axes[0], max_num_features=10, importance_type='gain', title='LGBM Engine', color='teal')
pd.Series(final_xgb.feature_importances_, index=features_list).nlargest(10).sort_values().plot(kind='barh', ax=axes[1], color='steelblue', title='XGB Engine')
pd.Series(final_cat.get_feature_importance(), index=features_list).nlargest(10).sort_values().plot(kind='barh', ax=axes[2], color='crimson', title='CatBoost Engine')
plt.suptitle('Model Architecture Logic', y=1.05)
plt.tight_layout()
plt.savefig("01_feature_importance.png", dpi=150, bbox_inches='tight')

# B. SHAP Global Summary
print("   -> Calculating SHAP Values (Takes a few seconds)...")
explainer = shap.TreeExplainer(final_lgbm)
shap_values = explainer(X_test)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("SHAP Global: What drives the AI's Decisions?", pad=20)
plt.tight_layout()
plt.savefig("02_shap_global.png", dpi=150, bbox_inches='tight')
plt.close()

# C. SHAP Local Waterfall (Peak Hour)
peak_idx = np.argmax(test_preds)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[peak_idx], show=False)
plt.title(f"SHAP Local: Analyzing the Highest Prediction ({test_dates.iloc[peak_idx]})", pad=20)
plt.tight_layout()
plt.savefig("03_shap_peak_event.png", dpi=150, bbox_inches='tight')
plt.close()

print("✅ Assets Saved: 'FINAL_SUBMISSION_2021.csv', '01_feature_importance.png', '02_shap_global.png', '03_shap_peak_event.png'")
print("🚀 SCRIPT COMPLETED! GO WIN THAT HACKATHON!")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm # Loop ki progress bar ke liye

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("\n" + "🌀"*35)
print("🌀 THE AUTOREGRESSIVE (RECURSIVE) EXPERIMENT 🌀")
print("🌀"*35)

# =========================================================
# 1. LOAD DATA & TRAIN FEATURES (HORIZON = 1 HOUR)
# =========================================================
print("\n📦 1. Loading & Engineering Features (1-Hour Ahead)...")
df = pd.read_csv("cleaned_energy_data_model2.csv")
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.sort_values('start_time').reset_index(drop=True)

cons = df['consumption']

# A. Calendar Features
df['hour']        = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.dayofweek
df['month']       = df['start_time'].dt.month
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# B. Lags (Horizon = 1, so lag_1h is allowed for TRAINING ONLY)
df['lag_1h']   = cons.shift(1)
df['lag_24h']  = cons.shift(24)
df['lag_168h'] = cons.shift(168)

# C. Rolling Features
df['rolling_mean_24h'] = cons.shift(1).rolling(24).mean()
df['rolling_std_24h']  = cons.shift(1).rolling(24).std()

features_list = ['hour', 'day_of_week', 'month', 'is_weekend',
                 'lag_1h', 'lag_24h', 'lag_168h',
                 'rolling_mean_24h', 'rolling_std_24h']

# Drop NaN rows caused by shift(168)
df_clean = df.dropna(subset=features_list + ['consumption']).copy()

train_mask = df_clean['start_time'].dt.year <= 2019
val_mask   = df_clean['start_time'].dt.year == 2020
test_mask  = df_clean['start_time'].dt.year == 2021

X_train_base, y_train_base = df_clean.loc[train_mask, features_list], df_clean.loc[train_mask, 'consumption']
X_val, y_val               = df_clean.loc[val_mask, features_list], df_clean.loc[val_mask, 'consumption']
X_train_full, y_train_full = df_clean.loc[train_mask | val_mask, features_list], df_clean.loc[train_mask | val_mask, 'consumption']

# Test data mein hume sirf calendar features chahiye pehle se, baaki hum dynamically banayenge
X_test_calendar = df_clean.loc[test_mask, ['hour', 'day_of_week', 'month', 'is_weekend']]
y_test = df_clean.loc[test_mask, 'consumption'].values
test_dates = df_clean.loc[test_mask, 'start_time'].values

# =========================================================
# 2. FAST TUNING & TRAINING (Keeping it quick)
# =========================================================
print("\n🧠 2. Quick Tuning of Base Models...")
N_TRIALS = 5 # Very fast tuning just for the experiment

# LGBM
study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(lambda t: mean_absolute_error(y_val, lgb.LGBMRegressor(n_estimators=t.suggest_int('n', 300, 800), learning_rate=0.05, max_depth=6, random_state=42, verbose=-1).fit(X_train_base, y_train_base).predict(X_val)), n_trials=N_TRIALS)
final_lgbm = lgb.LGBMRegressor(n_estimators=study_lgbm.best_params['n'], learning_rate=0.05, max_depth=6, random_state=42, verbose=-1).fit(X_train_full, y_train_full)

# XGBoost
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(lambda t: mean_absolute_error(y_val, xgb.XGBRegressor(n_estimators=t.suggest_int('n', 300, 800), learning_rate=0.05, max_depth=5, random_state=42, tree_method='hist').fit(X_train_base, y_train_base, verbose=False).predict(X_val)), n_trials=N_TRIALS)
final_xgb = xgb.XGBRegressor(n_estimators=study_xgb.best_params['n'], learning_rate=0.05, max_depth=5, random_state=42, tree_method='hist').fit(X_train_full, y_train_full, verbose=False)

# Master Blender Weights (Equal weights for simplicity in this experiment)
W_LGBM = 0.5
W_XGB = 0.5
print("✅ Models trained! (Skipped CatBoost to make the loop faster)")

# =========================================================
# 3. THE AUTOREGRESSIVE LOOP (THE CORE EXPERIMENT)
# =========================================================
print("\n🔄 3. Starting the Autoregressive Loop for 2021 (8,760 hours)...")
print("   -> (This might take 1-2 minutes as it predicts one hour at a time)")

# We need the last 168 hours (1 week) of 2020 ACTUAL data to start the engine
history = list(y_train_full.values[-168:])
ar_predictions = []

# Loop over every single hour in 2021
for i in tqdm(range(len(X_test_calendar))):

    # 1. Get current hour's calendar features
    cal_feats = X_test_calendar.iloc[i].to_dict()

    # 2. DYNAMICALLY calculate lags and rolling stats from our 'history' list
    # Notice: We are using history[-1] which is the PREDICTION from the previous loop iteration!
    dyn_feats = {
        'lag_1h': history[-1],
        'lag_24h': history[-24],
        'lag_168h': history[-168],
        'rolling_mean_24h': np.mean(history[-24:]),
        'rolling_std_24h': np.std(history[-24:])
    }

    # Merge dictionaries and convert to DataFrame (needs to match model feature order)
    row_data = {**cal_feats, **dyn_feats}
    current_row = pd.DataFrame([row_data], columns=features_list)

    # 3. Predict the next hour
    pred_lgbm = final_lgbm.predict(current_row)[0]
    pred_xgb = final_xgb.predict(current_row)[0]

    final_pred = (W_LGBM * pred_lgbm) + (W_XGB * pred_xgb)

    # 4. APPEND TO HISTORY (The Recursive Step)
    ar_predictions.append(final_pred)
    history.append(final_pred)

ar_predictions = np.array(ar_predictions)

# =========================================================
# 4. EVALUATION: WITNESS THE DRIFT
# =========================================================
def print_metrics(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"{name:<18} | MAE: {mae:<7.2f} MWh | MAPE: {mape*100:<5.2f}%")

print("\n" + "="*60)
print("📉 AUTOREGRESSIVE EVALUATION (WATCH THE ERROR GROW)")
print("="*60)
print_metrics("First 24 Hours", y_test[:24], ar_predictions[:24])
print_metrics("First 1 Week",   y_test[:168], ar_predictions[:168])
print_metrics("First 1 Month",  y_test[:720], ar_predictions[:720])
print("-" * 60)
print_metrics("FULL YEAR 2021", y_test, ar_predictions)
print("="*60)

# =========================================================
# 5. VISUALIZING THE DRIFT
# =========================================================
print("\n📸 Generating Drift Visualization...")
plt.figure(figsize=(15, 8))

# Plot first 2 months to clearly see where it loses track
plot_horizon = 24 * 60 # 60 days
plt.plot(test_dates[:plot_horizon], y_test[:plot_horizon], label='Actual Reality', color='black', alpha=0.6)
plt.plot(test_dates[:plot_horizon], ar_predictions[:plot_horizon], label='Autoregressive Fake Reality', color='red', linewidth=1.5)

plt.axvline(x=test_dates[168], color='blue', linestyle='--', label='1 Week Mark')
plt.axvline(x=test_dates[720], color='orange', linestyle='--', label='1 Month Mark')

plt.title("Autoregressive Drift: How a small error compounds over time", fontsize=14)
plt.ylabel("Energy Consumption (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("autoregressive_drift.png", dpi=150)
print("✅ Saved plot to 'autoregressive_drift.png'")
print("Experiment complete! Check the terminal output to see how the MAPE increases.")

In [ ]:
!pip install optuna catboost